# Генерация синтетических данных для A/B теста

## **Задача:** оценить эффект от изменения верстки раздела «Автозапчасти» на Фарпосте - переход от списка объявлений к плиточному виду.

### Поскольку реального трафика у нас нет, мы генерируем синтетические данные. Чтобы параметры генерации были реалистичными, а не взятыми из воздуха, предварительно был проведен анализ датасета **Avito Context Ad Clicks** (категория 34 - Автозапчасти) и получены следующие ориентиры:

| Параметр | Значение | Как получено |
|---|---|---|
| Уникальных пользователей в день | ~3 500 (медиана) | `search_info`, кат. 34 |
| Медианное время сессии | ~7 мин (420 сек) | `search_info` + `visits_stream` |
| Просмотров объявлений за сессию | ~3 (нижняя оценка) | `train_search_stream` |
| CTR (базовый) | ~15% | литературная норма для маркетплейсов |

## **Почему Авито, а не Фарпост?** 
### Аналитика Фарпост закрыта. Avito Context Ad Clicks - публичный датасет со схожей структурой категорий и что также важно - он содержит реальные, а не синтетические данные о поведении пользователей. Абсолютные цифры трафика уменьшаем (~в 10 раз) с учётом регионального масштаба Фарпоста. Качественные паттерны (форма распределений, соотношение кликов к просмотрам) переносим без изменений


In [1]:
from datetime import date, timedelta
import pandas as pd
import numpy as np
import random
import os

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

## 1. Параметры генерации

### Числовые параметры выведены из анализа датасета Авито (см. `preprocessing.ipynb`)

### Длительность теста - 21 день

### Это сделано для того чобы:
#### - захватить оба паттерна недели (будни/выходные)
#### - дать достаточно наблюдений для статистической мощности
#### - избежать эффекта при котором в первые дни пользователи реагируют на новинку сильнее

### Трафик - 150–350 сессий в день

#### На Авито в категории 34 медиана уникальных пользователей в день ~3 500. Фарпост - региональная, а не федеральная платформа, поэтому масштабируем в ~10 раз вниз и получаем ~350. Добавляем случайность в 50% чтобы имитировать реальную вариативность трафика (выходные, праздники, колебания спроса)

### Время на странице — медиана 7 мин, mean ~21 мин

#### На Авито медиана сессии в автозапчастях - 7 минут, среднее - 21 минута. Большое расхождение нормально, так как небольшая доля пользователей проводит очень много времени (сравнивают много вариантов) и тянет среднее вверх. Генерируем из логнормального распределения, потому что оно естественно воспроизводит этот «длинный хвост» и всегда даёт положительные значения

### Просмотры объявлений - loc=12 для контроля

#### Авито показывает ~3 объявления на поисковый запрос (при частичном логировании - только позиции 1, 2, 6, 7, 8). Реальное число выше. Пользователь делает несколько поисков за сессию и оцениваем ~10–15 просмотров. Берём `loc=12` для контрольной группы (список), `loc=18` для тестовой (плитка): плитка компактнее, за тот же скролл видно на ~50% больше карточек

### CTR - 15% базовый, 18% в тесте

#### 15% - типичное значение CTR для маркетплейсов
### Гипотеза теста: плиточный вид улучшает CTR на ~3 процентных пункта
#### Это умеренный, реалистичный эффект - не слишком маленький чтобы было неинтересно, не слишком большой чтобы выглядеть фантастически.

In [2]:
# Временной диапазон
TEST_START = date(2026, 5, 1)
TEST_DAYS  = 21

# Трафик (сессий в день)
# Авито кат.34: медиана ~3500 уников/день.
# Фарпост — региональная платформа, делим на ~10 → ~350.
# Нижняя граница 150 — «тихие» дни (выходные, нет новых объявлений).
SESSIONS_PER_DAY_MIN = 150
SESSIONS_PER_DAY_MAX = 350

# Время на странице (секунды)
# Авито: медиана ~420 сек, mean ~1260 сек.
# Логнормальное распределение воспроизводит длинный правый хвост.
# Параметры подобраны так чтобы медиана была примено 420 сек:
# median = exp(mu) -> mu = ln(420) = примерно 6.04
# sigma = 1.0 даёт разброс, близкий к наблюдаемому на Авито
TIME_MU_CONTROL = 6.04   # ln(420)
TIME_MU_TEST = 6.15   # ln(470) - плитка чуть дольше удерживает внимание
TIME_SIGMA = 1.0
TIME_CLIP_MIN = 10     # отсекаем «случайные» заходы < 10 сек
TIME_CLIP_MAX = 10800  # 3 часа - верхняя граница из preprocessing

# Просмотры объявлений
# Авито: ~3 объявления на поисковый запрос (частичный лог)
# Несколько запросов за сессию -> оцениваем ~12 для списка.
# Плитка: за тот же скролл ~50% больше карточек -> loc=18.
VIEWED_LOC_CONTROL = 12
VIEWED_LOC_TEST = 18
VIEWED_SCALE = 5    # стандартное отклонение
VIEWED_CLIP_MIN = 1
VIEWED_CLIP_MAX = 100

# CTR
# 15% базовый - норма для маркетплейса
# Гипотеза: плитка улучшает CTR на ~3 п.п. -> 18%.
CTR_CONTROL = 0.15
CTR_TEST = 0.18


## 3. Генерация дат и идентификаторов

### Генерируем сессии день за днём со случайным количеством в каждый день.

### Это важно: в реальном трафике нет идеально равномерного потока —выходные дни, новые объявления, сезонность дают колебания.

### `session_id` уникален для каждой сессии.
### `user_id` тоже уникален на сессию — мы упрощаем и не моделируем возвращающихся пользователей (один пользователь = одна сессия)


In [3]:
dates = []
for i in range(TEST_DAYS):
    n_sessions = random.randint(SESSIONS_PER_DAY_MIN, SESSIONS_PER_DAY_MAX)
    dates.extend([TEST_START + timedelta(days=i)] * n_sessions)

N = len(dates)
print(f'Всего сессий: {N}')
print(f'Период: {TEST_START} — {TEST_START + timedelta(days=TEST_DAYS - 1)}')
print(f'Среднее сессий в день: {N / TEST_DAYS:.0f}')

session_ids = random.sample(range(100_000, 999_999), N)
user_ids    = random.sample(range(100_000, 999_999), N)


Всего сессий: 4908
Период: 2026-05-01 — 2026-05-21
Среднее сессий в день: 234


## 4. Сплитирование пользователей

### Распределяем пользователей между группами случайно в соотношении 50/50

### **Критерии сплитирования:**
#### - Рандомизируем по **пользователю** (не сессии). В нашем упрощении каждая строка = уникальный пользователь, поэтому рандомизация по строке эквивалентна рандомизации по пользователю.
#### - Соотношение 50/50 максимизирует статистическую мощность при заданном общем размере выборки.
#### - После генерации проверим SRM (Sample Ratio Mismatch) — что группы действительно получились ~равными.

In [5]:
groups = np.random.choice(['test', 'control'], size=N, p=[0.5, 0.5])

# SRM check: убеждаемся что сплит близок к 50/50
n_test = (groups == 'test').sum()
n_control = (groups == 'control').sum()
print(f'Test: {n_test} ({n_test/N*100:.1f}%)')
print(f'Control: {n_control} ({n_control/N*100:.1f}%)')
print(f'Отклонение от 50/50: {abs(n_test - n_control)} сессий ({abs(n_test/N - 0.5)*100:.2f} п.п.)')


Test: 2524 (51.4%)
Control: 2384 (48.6%)
Отклонение от 50/50: 140 сессий (1.43 п.п.)


## 5. Датасет без эффекта (нулевая гипотеза)

### Генерируем данные где обе группы одинаковы - плитка не отличается от списка. Этот датасет нужен чтобы показать поведение статистического теста в ситуации когда H0 верна: тест должен **не** отвергнуть нулевую гипотезу

### Все параметры (время, просмотры, CTR) одинаковы для test и control. Разница между группами - только случайный шум.

### **Важно:** используется `np.random.binomial(n=offers_viewed, p=CTR_CONTROL)` для кликов. Это гарантирует что `offers_clicked <= offers_viewed` по построению пользователь не может кликнуть больше раз чем видел объявлений


In [8]:
# Время на странице — одинаковое для обеих групп
time_null = (np.random.lognormal(mean=TIME_MU_CONTROL, sigma=TIME_SIGMA, size=N)
             .clip(TIME_CLIP_MIN, TIME_CLIP_MAX)
             .astype(int))

# Просмотры объявлений — одинаковые для обеих групп
viewed_null = (np.random.normal(loc=VIEWED_LOC_CONTROL, scale=VIEWED_SCALE, size=N)
               .clip(VIEWED_CLIP_MIN, VIEWED_CLIP_MAX)
               .astype(int))

# Клики: binomial от числа просмотров с одинаковым CTR
# binomial(n, p) гарантирует 0 <= clicks <= n
clicked_null = np.random.binomial(n=viewed_null, p=CTR_CONTROL)

df_null = pd.DataFrame({
    'session_id': session_ids,
    'user_id': user_ids,
    'group': groups,
    'date': dates,
    'time_on_page': time_null,
    'offers_viewed_per_session': viewed_null,
    'offers_clicked_per_session': clicked_null,
})

print('Датасет без эффекта:')
print(df_null.groupby('group')[['time_on_page','offers_viewed_per_session','offers_clicked_per_session']].mean().round(2))
df_null.head()


Датасет без эффекта:
         time_on_page  offers_viewed_per_session  offers_clicked_per_session
group                                                                       
control        689.72                      11.65                        1.74
test           653.96                      11.66                        1.76


,session_id,user_id,group,date,time_on_page,offers_viewed_per_session,offers_clicked_per_session
0,629903,761393,control,2026-05-01,1450,5,0
1,731262,941134,control,2026-05-01,278,14,2
2,127824,670065,control,2026-05-01,251,21,3
3,688508,933456,control,2026-05-01,275,22,2
4,308496,120478,control,2026-05-01,60,9,0


## 6. Датасет с эффектом (альтернативная гипотеза)

### Генерируем данные где плитка (test) действительно лучше списка (control). Этот датасет нужен чтобы показать что статистический тест обнаруживает реальный эффект когда он есть.

### **Заложенные различия между группами:**

| Метрика | Control (список) | Test (плитка) | Обоснование |
|---|---|---|---|
| `time_on_page` | медиана ~420 сек | медиана ~470 сек | плитка нагляднее, дольше изучают |
| `offers_viewed` | loc=12 | loc=18 | компактный формат, больше карточек на экране |
| CTR | 15% | 18% | основная гипотеза теста |

### **Почему клики генерируются от `offers_viewed_effect`, а не от `offers_viewed_null`:**
#### В тестовой группе пользователь видит больше объявлений (18 vs 12). Если бы мы использовали базовые просмотры, мы бы занизили абсолютное число кликов и исказили картину. CTR (доля кликов) задаётся через `p` в binomial, а абсолютный объём - через `n` (число просмотров).


In [9]:
# Время: test группа проводит чуть больше времени (плитка удерживает внимание)
time_effect = np.where(
    groups == 'test',
    np.random.lognormal(mean=TIME_MU_TEST,    sigma=TIME_SIGMA, size=N),
    np.random.lognormal(mean=TIME_MU_CONTROL, sigma=TIME_SIGMA, size=N)
).clip(TIME_CLIP_MIN, TIME_CLIP_MAX).astype(int)

# Просмотры: test видит больше карточек за сессию
viewed_effect = np.where(
    groups == 'test',
    np.random.normal(loc=VIEWED_LOC_TEST,     scale=VIEWED_SCALE, size=N),
    np.random.normal(loc=VIEWED_LOC_CONTROL,  scale=VIEWED_SCALE, size=N)
).clip(VIEWED_CLIP_MIN, VIEWED_CLIP_MAX).astype(int)

# Клики: разные CTR для групп, от соответствующих просмотров
# Важно: используем viewed_effect (не viewed_null) — клики зависят от просмотров в этой же группе
clicked_effect = np.where(
    groups == 'test',
    np.random.binomial(n=viewed_effect, p=CTR_TEST),
    np.random.binomial(n=viewed_effect, p=CTR_CONTROL)
)

df_effect = pd.DataFrame({
    'session_id': session_ids,
    'user_id': user_ids,
    'group': groups,
    'date': dates,
    'time_on_page': time_effect,
    'offers_viewed_per_session': viewed_effect,
    'offers_clicked_per_session': clicked_effect,
})

print('Датасет с эффектом:')
print(df_effect.groupby('group')[['time_on_page','offers_viewed_per_session','offers_clicked_per_session']].mean().round(2))
df_effect.head()


Датасет с эффектом:
         time_on_page  offers_viewed_per_session  offers_clicked_per_session
group                                                                       
control        704.56                      11.66                        1.77
test           776.87                      17.60                        3.16


,session_id,user_id,group,date,time_on_page,offers_viewed_per_session,offers_clicked_per_session
0,629903,761393,control,2026-05-01,201,13,2
1,731262,941134,control,2026-05-01,218,12,4
2,127824,670065,control,2026-05-01,413,14,1
3,688508,933456,control,2026-05-01,447,7,1
4,308496,120478,control,2026-05-01,1581,16,2


## 7. Проверка метрик по группам

### Считаем все метрики с разбивкой по группам, так как именно это нужно для статистического теста. Считать метрику по всему датасету без разбивки на test/control бессмысленно: мы бы сравнивали два датасета целиком вместо того чтобы сравнивать группы внутри одного датасета.

### **Метрики:**
#### - **CTR сессионный** - доля сессий с хотя бы одним кликом (основная метрика, не искажается размером плитки)
#### - **CTR по объявлениям** - среднее clicks/views по сессиям (дополнительная метрика, показывает эффективность карточки)
#### - **Среднее время на странице**
#### - **Среднее просмотров за сессию**

In [11]:
def compute_metrics(df, label):
    result = df.groupby('group').agg(
        sessions = ('session_id', 'count'),
        ctr_session = ('offers_clicked_per_session', lambda x: (x > 0).mean()),
        avg_time = ('time_on_page', 'mean'),
        avg_viewed = ('offers_viewed_per_session', 'mean'),
        avg_clicked = ('offers_clicked_per_session', 'mean'),
    ).round(3)

    # CTR по объявлениям считаем отдельно (деление двух столбцов)
    ctr_offer = (df.groupby('group')
                   .apply(lambda g: (g['offers_clicked_per_session'] /
                                     g['offers_viewed_per_session']).mean())
                   .rename('ctr_offer')
                   .round(3))
    result = result.join(ctr_offer)
    print(f'\n{label}')
    print(result.to_string())
    return result

metrics_null = compute_metrics(df_null, 'Без эффекта')
metrics_effect = compute_metrics(df_effect, 'С эффектом')



Без эффекта
         sessions  ctr_session  avg_time  avg_viewed  avg_clicked  ctr_offer
group                                                                       
control      2384        0.796   689.724      11.652        1.735      0.148
test         2524        0.795   653.963      11.659        1.764      0.149

С эффектом
         sessions  ctr_session  avg_time  avg_viewed  avg_clicked  ctr_offer
group                                                                       
control      2384        0.802   704.556      11.658        1.768      0.152
test         2524        0.946   776.869      17.595        3.156      0.179


C:\Users\w\AppData\Local\Temp\ipykernel_11312\2135059613.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g['offers_clicked_per_session'] /
C:\Users\w\AppData\Local\Temp\ipykernel_11312\2135059613.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g['offers_clicked_per_session'] /


## 8. Динамика по дням

### Смотрим как метрики ведут себя во времени. Это важно по двум причинам:

#### 1. **Эффект новизны:** в первые дни пользователи могут кликать чаще просто потому что видят что-то новое - не потому что плитка лучше. Если CTR тестовой группы падает к концу теста, это сигнал.

#### 2. **Технические проблемы:** резкий провал или скачок в конкретный день может означать баг в рандомизации, технический сбой на сайте или внешнее событие (акция, праздник).


In [12]:
daily = (df_effect
         .groupby(['date', 'group'])
         .agg(ctr_session=('offers_clicked_per_session', lambda x: (x > 0).mean()),
              avg_viewed=('offers_viewed_per_session', 'mean'))
         .round(3)
         .reset_index())

# Показываем первые и последние дни для контроля
print('Первые 3 дня:')
print(daily.head(6).to_string(index=False))
print('\nПоследние 3 дня:')
print(daily.tail(6).to_string(index=False))


Первые 3 дня:
      date   group  ctr_session  avg_viewed
2026-05-01 control        0.819      11.231
2026-05-01    test        0.935      17.359
2026-05-02 control        0.812      11.941
2026-05-02    test        0.946      17.441
2026-05-03 control        0.849      12.178
2026-05-03    test        0.928      17.747

Последние 3 дня:
      date   group  ctr_session  avg_viewed
2026-05-19 control        0.897      11.092
2026-05-19    test        0.907      17.860
2026-05-20 control        0.904      11.564
2026-05-20    test        0.955      17.919
2026-05-21 control        0.863      11.735
2026-05-21    test        0.944      17.056


## 9. Сохранение датасетов

### - `ab_null.csv` - нулевая гипотеза (эффекта нет)
### - `ab_effect.csv` - альтернативная гипотеза (плитка лучше)


In [13]:
os.makedirs('data', exist_ok=True)

df_null.to_csv('data/ab_null.csv',   index=False)
df_effect.to_csv('data/ab_effect.csv', index=False)

print(f'ab_null.csv:   {df_null.shape[0]} строк, {df_null.shape[1]} столбцов')
print(f'ab_effect.csv: {df_effect.shape[0]} строк, {df_effect.shape[1]} столбцов')
print('\nСтолбцы:', list(df_null.columns))


ab_null.csv:   4908 строк, 7 столбцов
ab_effect.csv: 4908 строк, 7 столбцов

Столбцы: ['session_id', 'user_id', 'group', 'date', 'time_on_page', 'offers_viewed_per_session', 'offers_clicked_per_session']


## 10. Заключение

### Данные сгенерированы намеренно с заложенным эффектом в `ab_effect.csv` и без эффекта в `ab_null.csv`

### Мы проверяем, что:

#### 1. Тест **обнаруживает** эффект когда он реально есть (`ab_effect.csv`)
#### 2. Тест **не ложно срабатывает** когда эффекта нет (`ab_null.csv`)

### В реальном A/B тесте мы бы не знали заранее есть ли эффект. Данные собирались бы с живого трафика Фарпоста, и задача бы заключалась в определении есть ли статистически значимое различие между группами